# 02 - Silver transform

Membersihkan bronze, menjalankan DQ, dan menulis silver Delta.

In [ ]:
window_start = "2015-10-01"
window_end = "2015-10-31"

In [ ]:
# Parameters
window_start = "2015-10-01"
window_end = "2015-10-31"

# Configuration
from datetime import datetime, timezone
import os

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    DoubleType,
    MapType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

STORAGE_ACCOUNT = "sthidrogkiq7v7oc3lns"
BRONZE = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"
SILVER = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net"
TIMEZONE = "America/Los_Angeles"
RUN_ID = datetime.now(timezone.utc).strftime("silver-%Y%m%dT%H%M%SZ")

# Delta setup
spark.conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
spark.conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

METEO_VARIABLES = [
    "precipitation",
    "rain",
    "soil_moisture_0_to_7cm",
    "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm",
    "temperature_2m",
    "relative_humidity_2m",
    "surface_pressure",
    "et0_fao_evapotranspiration",
    "wind_speed_10m",
    "shortwave_radiation",
    "cloud_cover",
]

USGS_PROPERTIES = StructType([
    StructField("time_series_id", StringType()),
    StructField("monitoring_location_id", StringType()),
    StructField("parameter_code", StringType()),
    StructField("statistic_id", StringType()),
    StructField("time", StringType()),
    StructField("value", StringType()),
    StructField("unit_of_measure", StringType()),
    StructField("approval_status", StringType()),
    StructField("qualifier", StringType()),
    StructField("last_modified", StringType()),
    StructField("method_category", StringType()),
])

USGS_COLLECTION = StructType([
    StructField("features", ArrayType(StructType([
        StructField("id", StringType()),
        StructField("properties", USGS_PROPERTIES),
    ]))),
])

METEO_COLLECTION = StructType([
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("elevation", DoubleType()),
    StructField("timezone", StringType()),
    StructField("hourly_units", MapType(StringType(), StringType())),
    StructField("hourly", StructType(
        [StructField("time", ArrayType(StringType()))]
        + [StructField(variable, ArrayType(DoubleType())) for variable in METEO_VARIABLES]
    )),
])

PATH_REGEX_STATION = r"station_id=([^/]+)"
PATH_REGEX_DATE = r"ingest_date=([^/]+)"


def path_columns(frame):
    return (
        frame.withColumn("_file", F.input_file_name())
        .withColumn("station_id", F.regexp_extract("_file", PATH_REGEX_STATION, 1))
        .withColumn("ingest_date", F.regexp_extract("_file", PATH_REGEX_DATE, 1))
    )


def dedupe(frame, keys):
    window = Window.partitionBy(*keys).orderBy(F.col("last_modified_raw").desc_nulls_last())
    return (
        frame.withColumn("_row_number", F.row_number().over(window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )


def add_local_time(frame, utc_column="time_utc"):
    return (
        frame.withColumn("time_utc", F.col(utc_column))
        .withColumn("time_local", F.from_utc_timestamp(F.col("time_utc"), TIMEZONE))
        .withColumn("date_local", F.date_format("time_local", "yyyy-MM-dd"))
        .withColumn("ingest_month", F.date_format("time_utc", "yyyy-MM"))
    )


def write_delta(frame, path, months):
    predicate = f"ingest_month >= '{months[0]}' AND ingest_month <= '{months[1]}'"
    writer = (
        frame.write.format("delta")
        .mode("overwrite")
        .partitionBy("station_id", "ingest_month")
    )
    try:
        from delta.tables import DeltaTable
        if DeltaTable.isDeltaTable(spark, path):
            writer = writer.option("replaceWhere", predicate)
    except Exception:
        pass
    writer.save(path)


def write_quarantine(frame, table):
    path = f"{SILVER}/_quarantine/table={table}/ingest_date={window_end}"
    frame.write.mode("overwrite").parquet(path)


def write_dq(rows, table):
    from pyspark.sql.types import LongType, StructField, StructType, StringType
    schema = StructType([
        StructField("table", StringType()),
        StructField("check", StringType()),
        StructField("failed_count", LongType()),
        StructField("run_id", StringType()),
    ])
    path = f"{SILVER}/_dq/table={table}/ingest_date={window_end}"
    spark.createDataFrame(rows, schema).write.mode("overwrite").parquet(path)


def month_bounds(start, end):
    return start[:7], end[:7]


def read_usgs(collection, parameter_code):
    path = (
        f"{BRONZE}/usgs/{collection}/parameter_code={parameter_code}/"
        "station_id=*/ingest_date=*/part-000.json"
    )
    raw = spark.read.schema(USGS_COLLECTION).json(path)
    return (
        path_columns(raw)
        .select(F.explode("features").alias("feature"), "station_id", "ingest_date")
        .select("feature.properties.*", "station_id", "ingest_date")
    )


def clean_continuous(collection, parameter_code, table):
    raw = read_usgs(collection, parameter_code)
    prepared = (
        raw.withColumnRenamed("value", "value_raw")
        .withColumn("value", F.col("value_raw").cast("double"))
        .withColumn("time_utc", F.to_timestamp("time"))
        .withColumnRenamed("time", "time_raw")
        .withColumnRenamed("last_modified", "last_modified_raw")
    )
    latest = dedupe(prepared, ["station_id", "parameter_code", "time_raw"])
    cleaned = add_local_time(
        latest.select(
            "station_id", "ingest_date", "time_utc", "time_raw", "parameter_code",
            "time_series_id", "monitoring_location_id", "statistic_id", "value", "value_raw",
            F.col("unit_of_measure").alias("unit"), "approval_status", "qualifier",
            "last_modified_raw", "method_category",
        )
    )
    invalid = (
        F.col("time_utc").isNull()
        | (F.col("value").isNull() & F.col("value_raw").isNotNull())
        | (F.col("value") < 0)
    )
    quarantine = cleaned.filter(invalid).select(
        "station_id", "ingest_date", "time_raw", "parameter_code", "value_raw",
        F.when(F.col("time_utc").isNull(), "invalid timestamp")
        .when(F.col("value") < 0, "negative value")
        .otherwise("non-numeric value").alias("reason"),
    )
    valid = cleaned.filter(~invalid).drop("time_raw", "last_modified_raw", "value_raw")
    months = month_bounds(window_start, window_end)
    write_delta(valid, f"{SILVER}/delta/usgs/continuous/parameter_code={parameter_code}", months)
    write_quarantine(quarantine, table)
    total = valid.count()
    write_dq([
        (table, "row_count", total, RUN_ID),
        (table, "null_time", valid.filter(F.col("time_utc").isNull()).count(), RUN_ID),
        (table, "null_value", valid.filter(F.col("value").isNull()).count(), RUN_ID),
        (table, "negative_value", valid.filter(F.col("value") < 0).count(), RUN_ID),
    ], table)
    print(table, "rows", total, "quarantine", quarantine.count())


def clean_daily():
    raw = read_usgs("daily", "00060")
    prepared = (
        raw.withColumnRenamed("value", "value_raw")
        .withColumn("value", F.col("value_raw").cast("double"))
        .withColumnRenamed("time", "date_raw")
        .withColumnRenamed("last_modified", "last_modified_raw")
    )
    latest = dedupe(prepared, ["station_id", "parameter_code", "date_raw"])
    cleaned = (
        latest.withColumn("date_local", F.date_format(F.to_date("date_raw"), "yyyy-MM-dd"))
        .withColumn("last_modified", F.to_timestamp("last_modified_raw"))
        .withColumn("ingest_month", F.substring("date_local", 1, 7))
    )
    invalid = F.col("date_local").isNull() | (F.col("value").isNull() & F.col("value_raw").isNotNull()) | (F.col("value") < 0)
    quarantine = cleaned.filter(invalid).select(
        "station_id", "ingest_date", "date_raw", "parameter_code", "value_raw",
        F.when(F.col("date_local").isNull(), "invalid date")
        .when(F.col("value") < 0, "negative value")
        .otherwise("non-numeric value").alias("reason"),
    )
    valid = cleaned.filter(~invalid).select(
        "station_id", "ingest_date", "date_local", "ingest_month", "parameter_code",
        "time_series_id", "monitoring_location_id", "statistic_id", "value",
        F.col("unit_of_measure").alias("unit"), "approval_status", "qualifier",
        "last_modified",
    )
    months = month_bounds(window_start, window_end)
    write_delta(valid, f"{SILVER}/delta/usgs/daily/parameter_code=00060", months)
    write_quarantine(quarantine, "usgs_daily_00060")
    total = valid.count()
    write_dq([
        ("usgs_daily_00060", "row_count", total, RUN_ID),
        ("usgs_daily_00060", "null_date", valid.filter(F.col("date_local").isNull()).count(), RUN_ID),
        ("usgs_daily_00060", "null_value", valid.filter(F.col("value").isNull()).count(), RUN_ID),
        ("usgs_daily_00060", "negative_value", valid.filter(F.col("value") < 0).count(), RUN_ID),
    ], "usgs_daily_00060")
    print("usgs_daily_00060 rows", total, "quarantine", quarantine.count())


def clean_meteo():
    path = f"{BRONZE}/open-meteo/station_id=*/ingest_date=*/part-000.json"
    raw = spark.read.schema(METEO_COLLECTION).json(path)
    arrays = [F.col("hourly.time").alias("time")] + [
        F.col(f"hourly.{variable}").alias(variable) for variable in METEO_VARIABLES
    ]
    exploded = (
        path_columns(raw)
        .select(
            "station_id", "ingest_date", "latitude", "longitude", "elevation",
            "timezone", F.explode(F.arrays_zip(*arrays)).alias("z"),
        )
    )
    cleaned = add_local_time(
        exploded.select(
            "station_id", "ingest_date", "latitude", "longitude", "elevation", "timezone",
            F.to_timestamp(F.col("z.time")).alias("time_utc"),
            *[F.col(f"z.{variable}").cast("double").alias(variable) for variable in METEO_VARIABLES],
        )
    )
    invalid = F.col("time_utc").isNull()
    quarantine = cleaned.filter(invalid).select(
        "station_id", "ingest_date", "time_utc", F.lit("invalid timestamp").alias("reason")
    )
    valid = cleaned.filter(~invalid)
    months = month_bounds(window_start, window_end)
    write_delta(valid, f"{SILVER}/delta/open-meteo", months)
    write_quarantine(quarantine, "open_meteo_hourly")
    total = valid.count()
    write_dq([
        ("open_meteo_hourly", "row_count", total, RUN_ID),
        ("open_meteo_hourly", "null_time", valid.filter(F.col("time_utc").isNull()).count(), RUN_ID),
    ], "open_meteo_hourly")
    print("open_meteo_hourly rows", total, "quarantine", quarantine.count())


clean_continuous("continuous", "00065", "usgs_continuous_00065")
clean_continuous("continuous", "00060", "usgs_continuous_00060")
clean_daily()
clean_meteo()
print("silver complete", RUN_ID)
